# 27 — Nuclear Receptor Weighted LGBM

Trains LGBM on combined PXR CRC train + ChEMBL NR bioactivity data
(PPARγ, FXR, RXRα, LXRα, VDR, PPARα) with phylogenetic sample weights.
NR1I subfamily (PXR, VDR) gets highest weight; distal NRs get lower.
Tests whether cross-target NR knowledge transfers to PXR.

Compound features: Morgan (2048) + RDKit (217).
All pEC50 values used as-is (ChEMBL IC50→pEC50 conversion already done).

In [1]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import lightgbm as lgb

from pxr.data import load_train, load_test
from pxr.chem import bemis_murcko
from pxr.eval import scaffold_kfold_indices, compute_metrics, rae as rae_fn
from pxr.featurize import combined, impute
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

SEED = 42; N_FOLDS = 5
print('Setup complete.')

Setup complete.


In [2]:
# ── 2. Load data ──────────────────────────────────────────────────────────────
train = load_train()
te    = load_test()
nr    = pd.read_parquet('../data/external/chembl_nr_targets.parquet')

print(f'CRC train: {len(train):,}  |  ChEMBL NR: {len(nr):,}  |  Test: {len(te):,}')
print(nr['target_name'].value_counts())

CRC train: 4,139  |  ChEMBL NR: 11,511  |  Test: 513
target_name
PPARg    4311
FXR      3187
RXRa     1364
LXRa     1175
PXR       947
VDR       523
PPARa       4
Name: count, dtype: int64


In [3]:
# ── 3. Assign phylogenetic sample weights ──────────────────────────────────────
# PXR=NR1I2; VDR=NR1I1 (same subfamily) get highest transfer
# FXR=NR1H4; LXRα=NR1H3 (NR1H subfamily) — moderate transfer
# RXRα=NR2B1 — partner receptor for PXR heterodimer: moderate
# PPARγ/α=NR1C — more distal
TARGET_WEIGHTS = {
    'PXR':   1.0,
    'VDR':   0.50,
    'FXR':   0.30,
    'LXRa':  0.25,
    'RXRa':  0.25,
    'PPARg': 0.15,
    'PPARa': 0.15,
}

nr['weight'] = nr['target_name'].map(TARGET_WEIGHTS).fillna(0.1)

# Filter: only IC50/Ki values with reasonable pEC50 range
nr_filt = nr[(nr['pec50'] >= 3.0) & (nr['pec50'] <= 10.0)].copy()
print(f'After quality filter: {len(nr_filt):,}')
print(nr_filt.groupby('target_name')['pec50'].describe().round(2))

After quality filter: 11,496
              count  mean   std   min   25%   50%   75%    max
target_name                                                   
FXR          3186.0  6.60  1.07  4.01  5.87  6.55  7.26  10.00
LXRa         1175.0  6.29  0.97  4.04  5.61  6.24  6.89   9.10
PPARg        4308.0  6.38  1.11  4.00  5.52  6.17  7.15  10.00
PXR           947.0  5.60  0.83  4.00  4.99  5.52  6.10   8.62
RXRa         1364.0  6.70  1.09  4.08  5.89  6.73  7.58   9.40
VDR           516.0  6.72  1.48  4.17  5.32  6.82  8.06   9.95


In [4]:
# ── 4. Featurize ──────────────────────────────────────────────────────────────
print('Featurizing CRC train...')
X_tr = impute(combined(train['smiles'].tolist()))
y_tr = train['pec50'].values

print('Featurizing ChEMBL NR...')
X_nr = impute(combined(nr_filt['smiles'].tolist()))
y_nr = nr_filt['pec50'].values
w_nr = nr_filt['weight'].values

print('Featurizing test...')
X_te = impute(combined(te['smiles'].tolist()))

print(f'Shapes — Train: {X_tr.shape}  NR: {X_nr.shape}  Test: {X_te.shape}')

Featurizing CRC train...


Featurizing ChEMBL NR...


Featurizing test...


Shapes — Train: (4139, 2265)  NR: (11496, 2265)  Test: (513, 2265)


In [5]:
# ── 5. Scaffold 5-fold CV on CRC train ────────────────────────────────────────
LGBM_PARAMS = dict(
    n_estimators=1200, num_leaves=64, learning_rate=0.04,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.2,
    min_child_samples=10, n_jobs=4, verbose=-1
)

scaffolds = train['smiles'].map(bemis_murcko).tolist()
splits    = scaffold_kfold_indices(scaffolds, n_splits=N_FOLDS, seed=SEED)

oof = np.full(len(y_tr), np.nan)
fold_metrics = []

for fold_i, (tr_idx, va_idx) in enumerate(splits):
    # CRC fold-train + all ChEMBL NR data
    X_fold = np.vstack([X_tr[tr_idx], X_nr])
    y_fold = np.concatenate([y_tr[tr_idx], y_nr])
    w_fold = np.concatenate([np.ones(len(tr_idx)), w_nr])

    m = lgb.LGBMRegressor(**LGBM_PARAMS)
    m.fit(X_fold, y_fold, sample_weight=w_fold)
    oof[va_idx] = m.predict(X_tr[va_idx])
    fold_rae = rae_fn(y_tr[va_idx], oof[va_idx])
    met = compute_metrics(y_tr[va_idx], oof[va_idx])
    fold_metrics.append(met)
    print(f'  Fold {fold_i+1}: RAE={fold_rae:.4f}  Spearman={met["Spearman"]:.4f}')

oof_rae = rae_fn(y_tr, oof)
cv_df   = pd.DataFrame(fold_metrics)
print(f'\nOOF RAE (global): {oof_rae:.4f}')
print(f'Mean fold RAE: {cv_df["RAE"].mean():.4f} +/- {cv_df["RAE"].std():.4f}')
print(f'\n  LGBM_base (PXR only):    ~0.575')
print(f'  LGBM_tuned (PXR only):   0.5394')
print(f'  NR-weighted (this):      {oof_rae:.4f}')

np.save(DATA_PROCESSED / 'oof_nr_weighted.npy', oof)

  Fold 1: RAE=0.5403  Spearman=0.7507


  Fold 2: RAE=0.6214  Spearman=0.6376


  Fold 3: RAE=0.6312  Spearman=0.6435


  Fold 4: RAE=0.5885  Spearman=0.6761


  Fold 5: RAE=0.6241  Spearman=0.6671

OOF RAE (global): 0.5964
Mean fold RAE: 0.6011 +/- 0.0378

  LGBM_base (PXR only):    ~0.575
  LGBM_tuned (PXR only):   0.5394
  NR-weighted (this):      0.5964


In [6]:
# ── 6. Full retrain + test ────────────────────────────────────────────────────
X_full = np.vstack([X_tr, X_nr])
y_full = np.concatenate([y_tr, y_nr])
w_full = np.concatenate([np.ones(len(y_tr)), w_nr])

final_m = lgb.LGBMRegressor(**LGBM_PARAMS)
final_m.fit(X_full, y_full, sample_weight=w_full)
te_preds = np.clip(final_m.predict(X_te), y_tr.min()-0.5, y_tr.max()+0.5)
np.save(DATA_PROCESSED / 'te_nr_weighted.npy', te_preds)

sub = pd.DataFrame({'Molecule Name': te['name'].values, 'SMILES': te['smiles'].values, 'pEC50': te_preds})
assert len(sub) == 513 and sub['pEC50'].notna().all()
out = SUBMISSIONS / '27_nr_weighted_lgbm.csv'
sub.to_csv(out, index=False)
print(f'Saved: {out}  |  OOF RAE: {oof_rae:.4f}')
print(sub['pEC50'].describe().round(3))

Saved: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\27_nr_weighted_lgbm.csv  |  OOF RAE: 0.5964
count    513.000
mean       4.849
std        0.624
min        2.751
25%        4.508
50%        4.968
75%        5.280
max        6.213
Name: pEC50, dtype: float64
